In [13]:
import pandas as pd
import numpy as np

# Load the original recovered datasets
users = pd.read_csv("Social_Engine_Users.csv")
posts = pd.read_csv("Social_Engine_Posts_Corrupted.csv")

print("Users dataset shape:", users.shape)
print("Posts dataset shape:", posts.shape)

Users dataset shape: (1500, 5)
Posts dataset shape: (12360, 8)


In [14]:
# Initial data audit

print("=== USERS DATASET ===")
print("\nColumns and data types:")
print(users.dtypes)

print("\nMissing values:")
print(users.isnull().sum())

print("\nDuplicate rows:", users.duplicated().sum())
print("Duplicate user IDs:", users["user_id"].duplicated().sum())


print("\n\n=== POSTS DATASET ===")
print("\nColumns and data types:")
print(posts.dtypes)

print("\nMissing values:")
print(posts.isnull().sum())

print("\nDuplicate rows:", posts.duplicated().sum())
print("Duplicate post IDs:", posts["post_id"].duplicated().sum())

=== USERS DATASET ===

Columns and data types:
user_id            object
location           object
language           object
account_created    object
follower_count      int64
dtype: object

Missing values:
user_id            0
location           0
language           0
account_created    0
follower_count     0
dtype: int64

Duplicate rows: 0
Duplicate user IDs: 0


=== POSTS DATASET ===

Columns and data types:
post_id          object
user_id          object
platform         object
text_content     object
timestamp        object
likes           float64
shares            int64
comments          int64
dtype: object

Missing values:
post_id            0
user_id            0
platform        1846
text_content    1746
timestamp          0
likes           1858
shares             0
comments           0
dtype: int64

Duplicate rows: 360
Duplicate post IDs: 360


In [15]:
# Inspect unique values, ranges, and suspicious patterns

print("=== PLATFORM VALUES ===")
print(posts["platform"].value_counts(dropna=False))

print("\n=== ENGAGEMENT STATISTICS ===")
print(posts[["likes", "shares", "comments"]].describe())

print("\nNegative likes:", (posts["likes"] < 0).sum())
print("Negative shares:", (posts["shares"] < 0).sum())
print("Negative comments:", (posts["comments"] < 0).sum())

print("\n=== SAMPLE TIMESTAMPS ===")
print(posts["timestamp"].head(20).to_string(index=False))

print("\n=== USERS CATEGORICAL VALUES ===")
print("\nLanguages:")
print(users["language"].value_counts())

print("\nLocations:")
print(users["location"].value_counts())

print("\n=== RELATIONSHIP CHECK ===")
invalid_users = ~posts["user_id"].isin(users["user_id"])
print("Posts referencing unknown user IDs:", invalid_users.sum())

=== PLATFORM VALUES ===
platform
YouTube      2136
Facebook     2135
Twitter      2119
Reddit       2086
Instagram    2038
NaN          1846
Name: count, dtype: int64

=== ENGAGEMENT STATISTICS ===
              likes        shares      comments
count  10502.000000  12360.000000  12360.000000
mean    2247.075795   1005.867557    504.081958
std     1797.342958    574.734542    288.796752
min    -4987.000000      0.000000      0.000000
25%     1038.250000    509.000000    252.000000
50%     2387.000000   1016.000000    503.000000
75%     3664.000000   1499.000000    755.000000
max     5000.000000   2000.000000   1000.000000

Negative likes: 525
Negative shares: 0
Negative comments: 0

=== SAMPLE TIMESTAMPS ===
         25-09-2024
         1722528840
2025-04-13T20:12:18
         10-09-2024
         31-05-2024
         24-01-2025
         1719394663
2025-03-27T13:44:32
2024-05-05T05:52:34
         11-02-2025
2024-06-11T14:54:47
         1736393335
         1725467245
         15-02-2025
  

In [16]:
# Create working copies so the recovered raw data remains untouched
users_clean = users.copy()
posts_clean = posts.copy()

print("Posts before duplicate removal:", len(posts_clean))

# Remove exact duplicate rows
posts_clean = posts_clean.drop_duplicates().copy()

print("Posts after duplicate removal:", len(posts_clean))
print("Duplicates removed:", len(posts) - len(posts_clean))

# Verify post_id uniqueness
print("Remaining duplicate post IDs:",
      posts_clean["post_id"].duplicated().sum())

Posts before duplicate removal: 12360
Posts after duplicate removal: 12000
Duplicates removed: 360
Remaining duplicate post IDs: 0


In [17]:
# Analyse corruption patterns before deciding how to repair them

print("=== MISSING VALUES AFTER DUPLICATE REMOVAL ===")
print(posts_clean.isnull().sum())

print("\n=== NEGATIVE LIKES ===")
print("Negative likes:", (posts_clean["likes"] < 0).sum())

# Check whether corruption types overlap
missing_platform = posts_clean["platform"].isna()
missing_text = posts_clean["text_content"].isna()
missing_likes = posts_clean["likes"].isna()
negative_likes = posts_clean["likes"] < 0

print("\n=== CORRUPTION OVERLAP ===")
print("Missing platform AND missing text:",
      (missing_platform & missing_text).sum())

print("Missing platform AND missing likes:",
      (missing_platform & missing_likes).sum())

print("Missing text AND missing likes:",
      (missing_text & missing_likes).sum())

print("Missing platform AND negative likes:",
      (missing_platform & negative_likes).sum())

print("Missing text AND negative likes:",
      (missing_text & negative_likes).sum())

print("\n=== ROWS BY NUMBER OF CORRUPTION FLAGS ===")

flags = pd.DataFrame({
    "missing_platform": missing_platform,
    "missing_text": missing_text,
    "missing_likes": missing_likes,
    "negative_likes": negative_likes
})

print(flags.sum(axis=1).value_counts().sort_index())

print("\n=== CORRUPTION BY PLATFORM ===")
print(
    posts_clean.groupby("platform", dropna=False)
    .agg(
        rows=("post_id", "count"),
        missing_likes=("likes", lambda x: x.isna().sum()),
        negative_likes=("likes", lambda x: (x < 0).sum())
    )
)

=== MISSING VALUES AFTER DUPLICATE REMOVAL ===
post_id            0
user_id            0
platform        1784
text_content    1688
timestamp          0
likes           1814
shares             0
comments           0
dtype: int64

=== NEGATIVE LIKES ===
Negative likes: 509

=== CORRUPTION OVERLAP ===
Missing platform AND missing text: 230
Missing platform AND missing likes: 260
Missing text AND missing likes: 266
Missing platform AND negative likes: 88
Missing text AND negative likes: 75

=== ROWS BY NUMBER OF CORRUPTION FLAGS ===
0    7073
1    4110
2     766
3      51
Name: count, dtype: int64

=== CORRUPTION BY PLATFORM ===
           rows  missing_likes  negative_likes
platform                                      
Facebook   2074            319              70
Instagram  1989            296              96
Reddit     2031            289              79
Twitter    2049            324              93
YouTube    2073            326              83
NaN        1784            260        

In [18]:
# Investigate whether missing/corrupted values can be reconstructed
# from IDs, users, or other observable patterns.

print("=== SAMPLE POST IDs ===")
print(posts_clean["post_id"].head(20).to_string(index=False))

print("\n=== SAMPLE USER IDs ===")
print(posts_clean["user_id"].head(20).to_string(index=False))

print("\n=== PLATFORM BY USER ===")

user_platform_counts = (
    posts_clean.dropna(subset=["platform"])
    .groupby("user_id")["platform"]
    .nunique()
)

print(user_platform_counts.describe())
print("\nUsers posting on only one known platform:",
      (user_platform_counts == 1).sum())

print("\n=== LIKES RELATIONSHIPS ===")
print(
    posts_clean[["likes", "shares", "comments"]]
    .corr()
)

print("\n=== VALID LIKES DISTRIBUTION ===")

valid_likes = posts_clean[
    posts_clean["likes"].notna() &
    (posts_clean["likes"] >= 0)
]["likes"]

print(valid_likes.describe())

print("\n=== NEGATIVE LIKE EXAMPLES ===")
print(
    posts_clean.loc[
        posts_clean["likes"] < 0,
        ["post_id", "platform", "likes", "shares", "comments"]
    ].head(15).to_string(index=False)
)

=== SAMPLE POST IDs ===
to64mgey2v3y
7f0wdauzbj89
dvvhg8eel45x
hgb9cxke4t7b
zwerpw3wk320
f85c8vm8ac8l
19ys654yi1if
5jywsz1bu0kp
ol0ai4w126p0
irjlew2pr8sw
cvdgyqy7xoq6
1coucvhys7md
b2or5rj5mmlt
tv6gljfonx0r
4xtny5n4bf70
k9tvoyfmg8az
cpiun4hvwfns
9nrn7caujskm
hr3cuhgraltd
ckz4ciuuw7c5

=== SAMPLE USER IDs ===
user_vfxs1pry
user_8l7rv5oe
user_nfo3ih5u
user_27aje6ur
user_9px5q0by
user_aedwfomw
user_27aje6ur
user_hvqqyzoi
user_634lriof
user_8v39sv2i
user_wv5x4t03
user_ouggehu2
user_8l7rv5oe
user_avt1fzq5
user_pywq9u0v
user_kf84zwv2
user_tb4fgpus
user_pvhgwztj
user_1tx7gaza
user_0snf4coc

=== PLATFORM BY USER ===
count    1500.000000
mean        3.734667
std         0.974114
min         1.000000
25%         3.000000
50%         4.000000
75%         4.000000
max         5.000000
Name: platform, dtype: float64

Users posting on only one known platform: 21

=== LIKES RELATIONSHIPS ===
             likes    shares  comments
likes     1.000000  0.010453  0.009679
shares    0.010453  1.000000  0.0

In [19]:
# Test whether negative likes appear to be sign-flipped valid values

negative_magnitudes = posts_clean.loc[
    posts_clean["likes"] < 0, "likes"
].abs()

print("=== NEGATIVE LIKE MAGNITUDES ===")
print(negative_magnitudes.describe())

print("\n=== VALID LIKE VALUES ===")
print(valid_likes.describe())

print("\nNegative magnitudes outside valid 0-5000 range:",
      ((negative_magnitudes < 0) |
       (negative_magnitudes > 5000)).sum())

print("\nMean comparison:")
print("Valid likes mean:", valid_likes.mean())
print("Negative magnitudes mean:", negative_magnitudes.mean())

print("\nMedian comparison:")
print("Valid likes median:", valid_likes.median())
print("Negative magnitudes median:", negative_magnitudes.median())

=== NEGATIVE LIKE MAGNITUDES ===
count     509.000000
mean     2460.677800
std      1430.096271
min        11.000000
25%      1205.000000
50%      2402.000000
75%      3640.000000
max      4987.000000
Name: likes, dtype: float64

=== VALID LIKE VALUES ===
count    9677.000000
mean     2493.505115
std      1438.286521
min         0.000000
25%      1240.000000
50%      2502.000000
75%      3724.000000
max      5000.000000
Name: likes, dtype: float64

Negative magnitudes outside valid 0-5000 range: 0

Mean comparison:
Valid likes mean: 2493.5051152216597
Negative magnitudes mean: 2460.6777996070728

Median comparison:
Valid likes median: 2502.0
Negative magnitudes median: 2402.0


In [20]:
# Repair sign-flipped negative like counts

negative_before = (posts_clean["likes"] < 0).sum()

# Negative engagement counts are invalid.
# Their absolute-value distribution closely matches the valid likes
# distribution and all magnitudes fall within the observed valid range
# of 0-5000. Therefore, they are treated as sign-flip corruption.
posts_clean.loc[
    posts_clean["likes"] < 0, "likes"
] = posts_clean.loc[
    posts_clean["likes"] < 0, "likes"
].abs()

negative_after = (posts_clean["likes"] < 0).sum()

print("Negative likes repaired:", negative_before)
print("Negative likes remaining:", negative_after)

Negative likes repaired: 509
Negative likes remaining: 0


In [21]:
# Standardize mixed timestamp formats
# Formats observed:
# 1. Unix epoch timestamps       -> 1722528840
# 2. ISO datetime strings        -> 2025-04-13T20:12:18
# 3. DD-MM-YYYY date strings     -> 25-09-2024

def parse_mixed_timestamp(value):
    value = str(value).strip()

    # Unix timestamp
    if value.isdigit():
        return pd.to_datetime(int(value), unit="s")

    # ISO datetime
    if "T" in value:
        return pd.to_datetime(value, format="%Y-%m-%dT%H:%M:%S")

    # DD-MM-YYYY
    return pd.to_datetime(value, format="%d-%m-%Y")


posts_clean["timestamp_clean"] = posts_clean["timestamp"].apply(
    parse_mixed_timestamp
)

print("Failed timestamp conversions:",
      posts_clean["timestamp_clean"].isna().sum())

print("\nOriginal vs standardized timestamps:")
print(
    posts_clean[
        ["timestamp", "timestamp_clean"]
    ].head(20).to_string(index=False)
)

print("\nTimestamp range:")
print("Earliest:", posts_clean["timestamp_clean"].min())
print("Latest:", posts_clean["timestamp_clean"].max())

Failed timestamp conversions: 0

Original vs standardized timestamps:
          timestamp     timestamp_clean
         25-09-2024 2024-09-25 00:00:00
         1722528840 2024-08-01 16:14:00
2025-04-13T20:12:18 2025-04-13 20:12:18
         10-09-2024 2024-09-10 00:00:00
         31-05-2024 2024-05-31 00:00:00
         24-01-2025 2025-01-24 00:00:00
         1719394663 2024-06-26 09:37:43
2025-03-27T13:44:32 2025-03-27 13:44:32
2024-05-05T05:52:34 2024-05-05 05:52:34
         11-02-2025 2025-02-11 00:00:00
2024-06-11T14:54:47 2024-06-11 14:54:47
         1736393335 2025-01-09 03:28:55
         1725467245 2024-09-04 16:27:25
         15-02-2025 2025-02-15 00:00:00
         1731343668 2024-11-11 16:47:48
         1740278648 2025-02-23 02:44:08
         14-12-2024 2024-12-14 00:00:00
         1716471784 2024-05-23 13:43:04
         1733293947 2024-12-04 06:32:27
         1732652943 2024-11-26 20:29:03

Timestamp range:
Earliest: 2024-05-01 00:00:00
Latest: 2025-04-30 21:57:10


In [22]:
# Investigate whether missing values show systematic patterns

print("=== MISSINGNESS RATES ===")
for col in ["platform", "text_content", "likes"]:
    print(
        f"{col}: "
        f"{posts_clean[col].isna().mean() * 100:.2f}%"
    )


print("\n=== MISSINGNESS BY USER LANGUAGE ===")

analysis_df = posts_clean.merge(
    users_clean[["user_id", "language"]],
    on="user_id",
    how="left"
)

missing_by_language = analysis_df.groupby("language").agg(
    posts=("post_id", "count"),
    platform_missing=("platform", lambda x: x.isna().mean()),
    text_missing=("text_content", lambda x: x.isna().mean()),
    likes_missing=("likes", lambda x: x.isna().mean())
)

print((missing_by_language * {
    "posts": 1,
    "platform_missing": 100,
    "text_missing": 100,
    "likes_missing": 100
}).round(2))


print("\n=== MISSING LIKES: OTHER ENGAGEMENT ===")

print("Rows WITH likes:")
print(
    posts_clean.loc[
        posts_clean["likes"].notna(),
        ["shares", "comments"]
    ].mean()
)

print("\nRows WITHOUT likes:")
print(
    posts_clean.loc[
        posts_clean["likes"].isna(),
        ["shares", "comments"]
    ].mean()
)


print("\n=== TEXT AVAILABILITY BY PLATFORM ===")
print(
    pd.crosstab(
        posts_clean["platform"].fillna("MISSING"),
        posts_clean["text_content"].isna(),
        normalize="index"
    ).round(3)
)

=== MISSINGNESS RATES ===
platform: 14.87%
text_content: 14.07%
likes: 15.12%

=== MISSINGNESS BY USER LANGUAGE ===
          posts  platform_missing  text_missing  likes_missing
language                                                      
ar         1123             15.23         13.54          14.78
de         1149             12.53         13.93          14.36
en         1220             15.41         14.18          15.66
es         1143             16.27         15.22          15.57
fr         1156             15.31         13.93          13.06
hi         1291             16.03         14.10          16.65
ja         1298             14.25         14.10          14.02
pt         1132             15.37         14.22          16.52
ru         1119             14.57         14.48          16.44
zh         1369             13.81         13.15          14.24

=== MISSING LIKES: OTHER ENGAGEMENT ===
Rows WITH likes:
shares      1006.621638
comments     503.500491
dtype: float64

Rows W

In [23]:
# Examine whether like distributions differ substantially by platform
# before deciding how missing likes should be handled.

valid_like_rows = posts_clean[
    posts_clean["likes"].notna()
]

likes_by_platform = valid_like_rows.groupby("platform")["likes"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

print("=== LIKES BY PLATFORM ===")
print(likes_by_platform.round(2))

print("\n=== OVERALL VALID LIKES ===")
print("Mean:", round(valid_like_rows["likes"].mean(), 2))
print("Median:", valid_like_rows["likes"].median())

=== LIKES BY PLATFORM ===
           count     mean  median      std  min     max
platform                                               
Facebook    1755  2528.86  2573.0  1461.10  0.0  5000.0
Instagram   1693  2500.93  2496.0  1419.37  1.0  5000.0
Reddit      1742  2488.11  2494.5  1438.55  3.0  5000.0
Twitter     1725  2437.69  2438.0  1448.50  1.0  4999.0
YouTube     1747  2517.77  2585.0  1408.56  4.0  5000.0

=== OVERALL VALID LIKES ===
Mean: 2491.86
Median: 2498.0


In [24]:
# Final missing-value handling

# Preserve unknown information explicitly rather than fabricating values.
posts_clean["platform"] = posts_clean["platform"].fillna("Unknown")
posts_clean["text_content"] = posts_clean["text_content"].fillna(
    "[MISSING CONTENT]"
)

# Missing likes are intentionally retained as NaN.
# No reliable relationship was found that could reconstruct them,
# and statistical imputation would introduce artificial values.

print("=== MISSING VALUES AFTER HANDLING ===")
print(posts_clean.isnull().sum())

print("\nPlatform values:")
print(posts_clean["platform"].value_counts())

print("\nMissing likes intentionally retained:",
      posts_clean["likes"].isna().sum())

print("\nMissing text markers:",
      (posts_clean["text_content"] == "[MISSING CONTENT]").sum())

=== MISSING VALUES AFTER HANDLING ===
post_id               0
user_id               0
platform              0
text_content          0
timestamp             0
likes              1814
shares                0
comments              0
timestamp_clean       0
dtype: int64

Platform values:
platform
Facebook     2074
YouTube      2073
Twitter      2049
Reddit       2031
Instagram    1989
Unknown      1784
Name: count, dtype: int64

Missing likes intentionally retained: 1814

Missing text markers: 1688


In [25]:
# Replace the inconsistent original timestamp representation
# with the validated standardized datetime.

posts_clean["timestamp"] = posts_clean["timestamp_clean"]
posts_clean = posts_clean.drop(columns=["timestamp_clean"])

print("=== FINAL DATA TYPES ===")
print(posts_clean.dtypes)

print("\nFinal shape:", posts_clean.shape)
print("Unique post IDs:", posts_clean["post_id"].nunique())
print("Duplicate rows:", posts_clean.duplicated().sum())
print("Negative likes:", (posts_clean["likes"] < 0).sum())

=== FINAL DATA TYPES ===
post_id                 object
user_id                 object
platform                object
text_content            object
timestamp       datetime64[ns]
likes                  float64
shares                   int64
comments                 int64
dtype: object

Final shape: (12000, 8)
Unique post IDs: 12000
Duplicate rows: 0
Negative likes: 0


In [26]:
# Merge cleaned post data with user information for EDA

analysis = posts_clean.merge(
    users_clean,
    on="user_id",
    how="left",
    validate="many_to_one"
)

print("Analysis dataset shape:", analysis.shape)

print("\nColumns:")
print(analysis.columns.tolist())

print("\nMissing user information:")
print(
    analysis[
        ["location", "language", "account_created", "follower_count"]
    ].isnull().sum()
)

print("\nSample:")
print(analysis.head())

Analysis dataset shape: (12000, 12)

Columns:
['post_id', 'user_id', 'platform', 'text_content', 'timestamp', 'likes', 'shares', 'comments', 'location', 'language', 'account_created', 'follower_count']

Missing user information:
location           0
language           0
account_created    0
follower_count     0
dtype: int64

Sample:
        post_id        user_id  platform  \
0  to64mgey2v3y  user_vfxs1pry    Reddit   
1  7f0wdauzbj89  user_8l7rv5oe    Reddit   
2  dvvhg8eel45x  user_nfo3ih5u   Unknown   
3  hgb9cxke4t7b  user_27aje6ur  Facebook   
4  zwerpw3wk320  user_9px5q0by    Reddit   

                                        text_content           timestamp  \
0  Bummed out with my new Air Max from Nike! Abso... 2024-09-25 00:00:00   
1  My one month review of Pepsi Crystal Pepsi: Hi... 2024-08-01 16:14:00   
2  Just unboxed my new Highlander from Toyota. Ex... 2025-04-13 20:12:18   
3  Comparing Pepsi Crystal Pepsi to the competiti... 2024-09-10 00:00:00   
4  My one week revie

In [27]:
# FINAL DATA VALIDATION
# Verify the cleaned datasets before beginning EDA

print("========== POSTS VALIDATION ==========")

# 1. Structure
print("\nShape:", posts_clean.shape)
print("Unique post IDs:", posts_clean["post_id"].nunique())
print("Duplicate rows:", posts_clean.duplicated().sum())
print("Duplicate post IDs:", posts_clean["post_id"].duplicated().sum())

# 2. Missing values
print("\nMissing values:")
print(posts_clean.isnull().sum())

# 3. Platform consistency
valid_platforms = {
    "Facebook", "Instagram", "Reddit",
    "Twitter", "YouTube", "Unknown"
}

invalid_platforms = ~posts_clean["platform"].isin(valid_platforms)

print("\nInvalid platform values:", invalid_platforms.sum())
print("Platform values:")
print(posts_clean["platform"].value_counts())

# 4. Engagement validity
print("\nNegative likes:",
      (posts_clean["likes"].dropna() < 0).sum())

print("Negative shares:",
      (posts_clean["shares"] < 0).sum())

print("Negative comments:",
      (posts_clean["comments"] < 0).sum())

print("Likes above 5000:",
      (posts_clean["likes"].dropna() > 5000).sum())

print("Shares above 2000:",
      (posts_clean["shares"] > 2000).sum())

print("Comments above 1000:",
      (posts_clean["comments"] > 1000).sum())

# 5. Timestamp validity
print("\nTimestamp type:", posts_clean["timestamp"].dtype)
print("Missing timestamps:", posts_clean["timestamp"].isna().sum())
print("Earliest timestamp:", posts_clean["timestamp"].min())
print("Latest timestamp:", posts_clean["timestamp"].max())

# 6. ID integrity
print("\nMissing post IDs:", posts_clean["post_id"].isna().sum())
print("Missing user IDs:", posts_clean["user_id"].isna().sum())

unknown_users = ~posts_clean["user_id"].isin(users_clean["user_id"])
print("Posts referencing unknown users:", unknown_users.sum())


print("\n========== USERS VALIDATION ==========")

print("\nShape:", users_clean.shape)
print("Unique user IDs:", users_clean["user_id"].nunique())
print("Duplicate rows:", users_clean.duplicated().sum())
print("Duplicate user IDs:", users_clean["user_id"].duplicated().sum())

print("\nMissing values:")
print(users_clean.isnull().sum())

print("\nNegative follower counts:",
      (users_clean["follower_count"] < 0).sum())

print("\nLanguages:")
print(sorted(users_clean["language"].unique()))

print("\n========== VALIDATION COMPLETE ==========")

========== POSTS VALIDATION ==========

Shape: (12000, 8)
Unique post IDs: 12000
Duplicate rows: 0
Duplicate post IDs: 0

Missing values:
post_id            0
user_id            0
platform           0
text_content       0
timestamp          0
likes           1814
shares             0
comments           0
dtype: int64

Invalid platform values: 0
Platform values:
platform
Facebook     2074
YouTube      2073
Twitter      2049
Reddit       2031
Instagram    1989
Unknown      1784
Name: count, dtype: int64

Negative likes: 0
Negative shares: 0
Negative comments: 0
Likes above 5000: 0
Shares above 2000: 0
Comments above 1000: 0

Timestamp type: datetime64[ns]
Missing timestamps: 0
Earliest timestamp: 2024-05-01 00:00:00
Latest timestamp: 2025-04-30 21:57:10

Missing post IDs: 0
Missing user IDs: 0
Posts referencing unknown users: 0

========== USERS VALIDATION ==========

Shape: (1500, 5)
Unique user IDs: 1500
Duplicate rows: 0
Duplicate user IDs: 0

Missing values:
user_id            0
loca

In [28]:
# Export final cleaned datasets

posts_clean.to_csv(
    "Social_Engine_Posts_Cleaned.csv",
    index=False
)

users_clean.to_csv(
    "Social_Engine_Users_Cleaned.csv",
    index=False
)

print("Cleaned datasets exported successfully.")
print("Posts:", posts_clean.shape)
print("Users:", users_clean.shape)

Cleaned datasets exported successfully.
Posts: (12000, 8)
Users: (1500, 5)
